In [ ]:
import math

# ==========================================
# 1. Minimax Algorithm with Leaf Counter
# ==========================================
minimax_leaf_evaluations = 0

def minimax(curDepth, nodeIndex, maxTurn, scores, targetDepth):
    global minimax_leaf_evaluations

    # Base case: target leaf depth reached
    if curDepth == targetDepth:
        minimax_leaf_evaluations += 1
        return scores[nodeIndex]

    if maxTurn:
        # Left child: 2 * nodeIndex, Right child: 2 * nodeIndex + 1
        return max(
            minimax(curDepth + 1, nodeIndex * 2, False, scores, targetDepth),
            minimax(curDepth + 1, nodeIndex * 2 + 1, False, scores, targetDepth)
        )
    else:
        return min(
            minimax(curDepth + 1, nodeIndex * 2, True, scores, targetDepth),
            minimax(curDepth + 1, nodeIndex * 2 + 1, True, scores, targetDepth)
        )

# ==========================================
# 2. Alpha-Beta Pruning with Leaf Counter
# ==========================================
alphabeta_leaf_evaluations = 0

def alphabeta(depth, index, maxTurn, scores, target, alpha, beta):
    global alphabeta_leaf_evaluations

    # Base case: target leaf depth reached
    if depth == target:
        alphabeta_leaf_evaluations += 1
        return scores[index]

    if maxTurn:
        value = float('-inf')
        for i in range(2):
            child_index = index * 2 + i  # i = 0 (left), i = 1 (right)
            value = max(value, alphabeta(depth + 1, child_index, False, scores, target, alpha, beta))
            alpha = max(alpha, value)
            if alpha >= beta:  # Beta cut-off
                break
        return value
    else:
        value = float('inf')
        for i in range(2):
            child_index = index * 2 + i  # i = 0 (left), i = 1 (right)
            value = min(value, alphabeta(depth + 1, child_index, True, scores, target, alpha, beta))
            beta = min(beta, value)
            if alpha >= beta:  # Alpha cut-off
                break
        return value

# ==========================================
# 3. Helper to Run and Compare
# ==========================================
def run_experiment(test_name, scores_input):
    global minimax_leaf_evaluations, alphabeta_leaf_evaluations

    # Pad to nearest power of 2 for a complete binary tree
    scores = list(scores_input)
    tree_depth = math.ceil(math.log2(len(scores)))
    scores += [0] * ((2 ** tree_depth) - len(scores))

    # Run Minimax
    minimax_leaf_evaluations = 0
    mm_result = minimax(0, 0, True, scores, tree_depth)

    # Run Alpha-Beta
    alphabeta_leaf_evaluations = 0
    ab_result = alphabeta(0, 0, True, scores, tree_depth, float('-inf'), float('inf'))

    pruned = minimax_leaf_evaluations - alphabeta_leaf_evaluations
    pct_reduction = (pruned / minimax_leaf_evaluations) * 100

    return {
        "name": test_name,
        "scores": scores,
        "mm_val": mm_result,
        "ab_val": ab_result,
        "mm_leaves": minimax_leaf_evaluations,
        "ab_leaves": alphabeta_leaf_evaluations,
        "reduction": pct_reduction
    }

# ==========================================
# 4. Test Cases (Lab Tasks 2 & 3)
# ==========================================
# Case A: Standard lab manual test case
case_a = [3, 5, 2, 9, 0, 7]

# Case B: Reordered for optimal pruning (best-case ordering)
# Arranging higher values to the left for MAX, lower values to the left for MIN
case_b = [9, 7, 5, 3, 2, 0, 4, 1]

# Case C: Reordered for worst-case pruning (searches nearly every leaf)
case_c = [1, 4, 0, 2, 3, 5, 7, 9]

results = [
    run_experiment("Case A (Lab Manual Input)", case_a),
    run_experiment("Case B (Best-case Ordering)", case_b),
    run_experiment("Case C (Worst-case Ordering)", case_c),
]

# ==========================================
# 5. Formatted Output Table
# ==========================================
print("=" * 86)
print(f"{'Test Case':<28} | {'Values (MM/AB)':<15} | {'Evaluations (MM vs AB)':<22} | {'Pruned %':<10}")
print("=" * 86)

for r in results:
    vals = f"{r['mm_val']} / {r['ab_val']}"
    leaves = f"{r['mm_leaves']} vs {r['ab_leaves']}"
    pct = f"{r['reduction']:.1f}%"
    print(f"{r['name']:<28} | {vals:<15} | {leaves:<22} | {pct:<10}")

print("=" * 86)

Test Case                    | Values (MM/AB)  | Evaluations (MM vs AB) | Pruned %  
Case A (Lab Manual Input)    | 5 / 5           | 8 vs 8                 | 0.0%      
Case B (Best-case Ordering)  | 5 / 5           | 8 vs 6                 | 25.0%     
Case C (Worst-case Ordering) | 5 / 5           | 8 vs 7                 | 12.5%     
